# Example 9.5.5 / 1: Capture and Replay a .tab Copy

Companion notebook for **9.5.5 Table Handler Diagnostics and Error Recovery** in the AMPL lab Documentation, section "Capture and Replay a `.tab` Copy". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Capture and Replay a `.tab` Copy

`table_debug` is a bit mask. Add the required values:

| Bit | Effect |
|---:|---|
| `1` | Make `read table` read the generated `.tab`-format file instead of the declared external table. If bit `2` is also set, bit `1` takes precedence. |
| `2` | Make `read table` write the external data it receives to a `.tab`-format file. |
| `4` | Make `write table` also write a `.tab`-format file. |
| `8` | Prevent `write table` from writing to the declared external table. |

The default is `0` (shown as an empty string by `option table_debug;`), which disables these
substitutions and captures. `table_debug_template` defaults to `%.dbtab`; AMPL replaces the
single `%` with the declared table name. The template must contain exactly one `%` and no
whitespace.

This example captures the input while reading the real file, deletes the file, replays the
capture, and finally writes a plan only to its capture:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param rate {PROD} > 0;
param profit {PROD};

table Products IN "amplcsv" "products.csv":
    PROD <- [product], rate, profit;
table Plan OUT "amplcsv" "plan.csv":
    {p in PROD} -> [product], profit[p] * rate[p] ~ per_hour;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import os
for name in ("plan.csv", "Plan.dbtab", "Products.dbtab"):
    if os.path.exists(name):
        os.remove(name)
with open("products.csv", "w") as f:
    f.write("product,rate,profit\nbands,200,25\ncoils,140,30\nplate,160,29\n")

In [5]:
# run.py
ampl.eval("load amplcsv.dll;")
ampl.eval("option table_debug 2;")
ampl.read_table("Products")
print(open("Products.dbtab").read())

os.remove("products.csv")
ampl.eval("reset data; option table_debug 1;")
ampl.read_table("Products")
print(ampl.param["rate"].to_dict())

ampl.eval("option table_debug 12;")
ampl.write_table("Plan")
print(os.path.exists("plan.csv"), os.path.exists("Plan.dbtab"))
ampl.eval("option table_debug 0;")

ampl.tab 1 2
product	rate	profit
bands	200	25
coils	140	30
plate	160	29

{'bands': 200, 'coils': 140, 'plate': 160}
False True
